In [2]:
import numpy as np
import sleap
import matplotlib.pyplot as plt
import sys, os
sys.path.append('../..')

In [3]:
from python.postprocess import *

In [5]:
original_dataset = sleap.load_file('/home/mingxiao/Desktop/jellyfish/video/video_1_clips/manual_5min_c0_copy.slp')
original_dataset

Labels(labeled_frames=9000, videos=1, skeletons=1, tracks=17)

In [13]:
def reconstruct_full_sequence(predicted_seqs, num_frames=9000):
    """
    Reconstructs the full sequence by averaging overlapping predictions.
    
    Args:
      predicted_seqs: numpy array of shape (num_seqs, seq_length, 17, 2)
      num_frames: total number of frames in the original data (e.g., 9000)
      seq_length: length of each sequence (e.g., 10)
    
    Returns:
      A numpy array of shape (num_frames, 17, 2) representing the improved points.
    """
    # Initialize an array to accumulate predictions and an array to count contributions.
    output = np.zeros((num_frames, 17, 2), dtype=np.float32)
    counts = np.zeros((num_frames, 1), dtype=np.float32)
    
    num_seqs = predicted_seqs.shape[0]
    seq_length = predicted_seqs.shape[1]
    for i in range(num_seqs):
        for j in range(seq_length):
            frame_idx = i + j
            if frame_idx < num_frames:
                output[frame_idx] += predicted_seqs[i, j]
                counts[frame_idx] += 1
    # Avoid division by zero and compute the average.
    output /= np.maximum(counts[:,:,None], 1)
    return output

In [14]:
results_affix = ['00', '0', '1', '2', '3', '4']
for affix in results_affix[3:]:
    print(f'affix: {affix}')
    denoised_coords = np.load(f'../results/denoised_coords_m{affix}.npy')
    print(denoised_coords.shape)
    denoised_coords = reconstruct_full_sequence(denoised_coords)
    print(denoised_coords.shape)
    assert denoised_coords.shape == (9000, 17, 2)
    predicted_dataset = dataset_with_new_points(original_dataset, denoised_coords, 
                                                node_name='tb', handle_first_frame=False, start_idx=0)
    predicted_dataset.save(f'../results/predicted_dataset_m{affix}.slp')

affix: 2
(8991, 10, 17, 2)
(9000, 17, 2)
Skeleton(description=None, nodes=[tb], edges=[], symmetries=[])
17
missing_pt_cnt: 0
affix: 3
(8996, 5, 17, 2)
(9000, 17, 2)
Skeleton(description=None, nodes=[tb], edges=[], symmetries=[])
17
missing_pt_cnt: 0
affix: 4
(8996, 5, 17, 2)
(9000, 17, 2)
Skeleton(description=None, nodes=[tb], edges=[], symmetries=[])
17
missing_pt_cnt: 0


In [15]:
all_predicted_pointss = []
for affix in results_affix:
    print(f'affix: {affix}')
    denoised_coords = np.load(f'../results/denoised_coords_m{affix}.npy')
    if denoised_coords.shape != (9000, 17, 2):
        denoised_coords = reconstruct_full_sequence(denoised_coords)
    print(denoised_coords.shape)
    all_predicted_pointss.append(denoised_coords)


affix: 00
(9000, 17, 2)
affix: 0
(9000, 17, 2)
affix: 1
(9000, 17, 2)
affix: 2
(9000, 17, 2)
affix: 3
(9000, 17, 2)
affix: 4
(9000, 17, 2)


In [16]:
y_true = coords = np.load('/home/mingxiao/Desktop/jellyfish/video/video_1_clips/manual_5min_c0_points.npy')
print(y_true.shape)

(9000, 17, 2)


In [18]:
for affix, predicted_points in zip(results_affix, all_predicted_pointss):
    print(f'm{affix}')
    diff = np.sum((y_true[:2500] - predicted_points[:2500]) ** 2) / (2500 * 17 * 2)
    print(diff)


m00
15.092387476456272
m0
8.124924370276473
m1
6.060766905063461
m2
13.149785789502848
m3
5.528012205329437
m4
5.740717705936208
